In [13]:
import pandas as pd
import os
import csv

In [14]:
BASE_PATH = "C:/Users/HOAI HUE/Desktop/tiki/data"

files = {
    "be_reviews": "tiki_reviews.csv",
    "be_listing": "tiki_me_be_products_listing.csv",
    "be_detail": "tiki_me_be_product_detail_full.csv",
    "be_product_id": "tiki_me_be_products_id.csv"
}

input_paths = {k: f"{BASE_PATH}/raw/{v}" for k, v in files.items()}
output_paths = {k: f"{BASE_PATH}/raw/Tiki_{k}.csv" for k in files.keys()}

In [15]:
def fix_broken_csv_lines(file_path):
    fixed_lines = []
    buffer = ""
    quote_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # cộng số dấu "
            quote_count += line.count('"')

            buffer += line.strip("\n")

            # nếu số dấu " là chẵn → dòng hoàn chỉnh
            if quote_count % 2 == 0:
                fixed_lines.append(buffer)
                buffer = ""
                quote_count = 0
            else:
                buffer += " "  # nối dòng

    return fixed_lines

In [16]:
def load_fixed_csv(file_path):
    lines = fix_broken_csv_lines(file_path)

    # convert thành dataframe
    from io import StringIO
    data = "\n".join(lines)

    df = pd.read_csv(
        StringIO(data),
        dtype=str,
        engine='python'
    )

    return df

In [17]:
def clean_text(df):
    for col in df.columns:
        df[col] = df[col].astype(str)

        df[col] = df[col].str.replace('\n', ' ', regex=False)
        df[col] = df[col].str.replace('\r', ' ', regex=False)
        df[col] = df[col].str.replace('"', '', regex=False)

        # KHÔNG replace dấu phẩy nữa (vì đã fix structure)
        df[col] = df[col].str.strip()

    return df

In [18]:
def fix_scientific(val):
    try:
        if isinstance(val, str) and ('E+' in val or 'e+' in val):
            return str(int(float(val)))
        return val
    except:
        return val


def apply_fix_scientific(df):
    for col in ['product_id', 'sku']:
        if col in df.columns:
            df[col] = df[col].apply(fix_scientific)
    return df

In [19]:
cleaned_data = {}

for name, path in input_paths.items():
    print(f"\n🚀 Processing {name}...")

    # FIX STRUCTURE
    df = load_fixed_csv(path)
    print(f"Loaded: {df.shape}")

    # CLEAN TEXT
    df = clean_text(df)

    # FIX NUMBER FORMAT
    df = apply_fix_scientific(df)

    cleaned_data[name] = df


🚀 Processing be_reviews...
Loaded: (74036, 16)

🚀 Processing be_listing...
Loaded: (5037, 5)

🚀 Processing be_detail...
Loaded: (1878, 28)

🚀 Processing be_product_id...
Loaded: (1864, 5)


In [21]:
#xem các giá trị duy nhất của category_lv3 trong tiki_me_be_products_id
print("\nUnique category_lv3 in product_id:")
print(cleaned_data['be_product_id']['category_lv3'].unique())


Unique category_lv3 in product_id:
<ArrowStringArray>
[                             'Tã dán',                   'Miếng lót sơ sinh',
                             'Tã quần',                             'Tã giấy',
                              'Tã vải',           'Tã bỉm dành cho người lớn',
                'Khăn giấy ướt cho bé',                      'Sữa bột cho bé',
        'Sữa công thức pha sẵn cho bé',                          'Bột ăn dặm',
                            'Vitamins',                'Nước trái cây cho bé',
                         'Bánh ăn dặm',       'Thực phẩm chế biến sẵn cho bé',
                       'Dầu ăn cho bé',             'Trà và Thức uống cho bé',
 'Các sản phẩm dinh dưỡng cho bé khác',                       'Gia vị cho bé',
                      'Mì, nui ăn dặm',              'Đồ dùng ăn uống cho bé',
              'Đồ dùng vệ sinh cho bé',            'Đồ dùng phòng ngủ cho bé',
                   'Đồ dùng bảo vệ bé',           'Dụng cụ chăm sóc sức khỏe

In [22]:
# Xóa tất cả thông tin của các category_lv3:'Tã bỉm dành cho người lớn', 'Dụng cụ chăm sóc sức khỏe','Combo đi sinh và gói quà tặng' trong clean_reviews, clean_listing,clean_detail,clean_product_id
categories_to_remove = ['Tã bỉm dành cho người lớn', 'Dụng cụ chăm sóc sức khỏe','Combo đi sinh và gói quà tặng']
for name in cleaned_data.keys():
    df = cleaned_data[name]
    if 'category_lv3' in df.columns:
        df = df[~df['category_lv3'].isin(categories_to_remove)]
        cleaned_data[name] = df
        print(f"After removing categories in {name}: {df.shape}")

    


After removing categories in be_product_id: (1620, 5)


In [23]:
def save_clean_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    df.to_csv(
        path,
        index=False,
        encoding='utf-8',
        quoting=csv.QUOTE_ALL   # 🔥 QUAN TRỌNG
    )

    print(f"✅ Saved: {path}")


for name, df in cleaned_data.items():
    save_clean_csv(df, output_paths[name])

✅ Saved: C:/Users/HOAI HUE/Desktop/tiki/data/raw/Tiki_be_reviews.csv
✅ Saved: C:/Users/HOAI HUE/Desktop/tiki/data/raw/Tiki_be_listing.csv
✅ Saved: C:/Users/HOAI HUE/Desktop/tiki/data/raw/Tiki_be_detail.csv
✅ Saved: C:/Users/HOAI HUE/Desktop/tiki/data/raw/Tiki_be_product_id.csv


In [24]:
for name, path in output_paths.items():
    print(f"\n🔍 Checking {name}")

    df = pd.read_csv(path, dtype=str)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))


🔍 Checking be_reviews
Shape: (74036, 16)
Columns: ['review_id', 'product_id', 'category_id', 'customer_id', 'customer_name', 'rating', 'title', 'content', 'created_at_ts', 'created_at_date', 'purchased_at_ts', 'purchased_at_date', 'days_since_purchase', 'helpful_count', 'is_verified', 'images_count']
  review_id product_id category_id customer_id    customer_name rating  \
0  20205835  278631185        2552     1166119  Mai Thị Lan Anh      5   
1  20181960  278631185        2552    29087580      Mai Ngọc Mi      5   

             title content created_at_ts created_at_date purchased_at_ts  \
0  Cực kì hài lòng     NaN    1768406964       1/14/2026      1766720336   
1  Cực kì hài lòng     NaN    1760546795      10/15/2025      1759377719   

  purchased_at_date days_since_purchase helpful_count is_verified images_count  
0        12/26/2025                  19             0        TRUE            0  
1         10/2/2025                  13             0        TRUE            0  

🔍